In [1]:
!pip install pandas nltk spacy contractions
!python -m nltk.downloader punkt_tab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 289.9/289.9 kB 5.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.3/118.3 kB 6.3 MB/s eta 0:00:00
<frozen runpy>:128: RuntimeWarning: 'nltk.downloader' found in sys.modules after import of package 'nltk', but prior to execution of 'nltk.downloader'; this may result in unpredictable behaviour
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [2]:
import os
base_dir = "/kaggle/working/"
os.chdir(base_dir)
os.makedirs("TextSummarization/data/raw", exist_ok=True)
os.makedirs("TextSummarization/data/processed", exist_ok=True)
os.makedirs("TextSummarization/preprocessing", exist_ok=True)

In [3]:
import shutil

# Define source and destination paths
input_dir = "/kaggle/input/text-summarization1/"
raw_dir = "TextSummarization/data/raw/"

# List of files to transfer
files = ["train.csv", "test.csv", "validation.csv"]

# Copy each file
for file in files:
    src = os.path.join(input_dir, file)
    dst = os.path.join(raw_dir, file)
    shutil.copy(src, dst)

print("Files successfully copied to raw directory.")


Files successfully copied to raw directory.


In [4]:
# normalize.py
with open("TextSummarization/preprocessing/normalize.py", "w") as f:
    f.write('''import re
import html
import nltk
from nltk.tokenize import sent_tokenize
import contractions
import pandas as pd

nltk.download('punkt_tab')

def clean_and_segment(text):
    if pd.isna(text): return ""
    text = contractions.fix(text)
    text = html.unescape(text)
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r"[^a-zA-Z0-9.,!?;:\\s]", "", text)
    text = re.sub(r'\\s+', ' ', text).strip()
    sentences = sent_tokenize(text)
    return ' '.join(sentences)
''')

# __init__.py
open("TextSummarization/preprocessing/__init__.py", "w").close()


In [5]:
with open("TextSummarization/main_preprocessing.py", "w") as f:
    f.write('''import os
import pandas as pd
from preprocessing.normalize import clean_and_segment
from tqdm import tqdm

# Ensure progress bar works with pandas apply
tqdm.pandas()

# Create directories if not exist
os.makedirs("TextSummarization/data/raw", exist_ok=True)
os.makedirs("TextSummarization/data/processed", exist_ok=True)

# Load 20,000 sample
df = pd.read_csv("TextSummarization/data/raw/train.csv").sample(n=20000, random_state=42)
df1 = pd.read_csv("TextSummarization/data/raw/validation.csv")
df2 = pd.read_csv("TextSummarization/data/raw/test.csv")

# Apply cleaning + segmentation with progress
print("🧹 Cleaning and segmenting text...")
df['clean_article'] = df['article'].progress_apply(clean_and_segment)
df['clean_summary'] = df['highlights'].progress_apply(clean_and_segment)
df1['clean_article'] = df1['article'].progress_apply(clean_and_segment)
df1['clean_summary'] = df1['highlights'].progress_apply(clean_and_segment)
df2['clean_article'] = df2['article'].progress_apply(clean_and_segment)
df2['clean_summary'] = df2['highlights'].progress_apply(clean_and_segment)

df.to_parquet("TextSummarization/data/processed/train_sample_50k.parquet", index=False)
df1.to_parquet("TextSummarization/data/processed/validation_sample.parquet", index=False)
df2.to_parquet("TextSummarization/data/processed/test_sample.parquet", index=False)

print("✅ Preprocessing complete and saved.")
''')


In [6]:
!python TextSummarization/main_preprocessing.py

[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
🧹 Cleaning and segmenting text...
100%|███████████████████████████████████| 11490/11490 [00:01<00:00, 9665.85it/s]
✅ Preprocessing complete and saved.


In [7]:
with open("TextSummarization/preprocessing/tokenizer.py", "w") as f:
    f.write('''import sentencepiece as spm
import pandas as pd
import numpy as np
import torch
from tqdm import tqdm

tqdm.pandas()

def train_sentencepiece(input_texts, model_prefix, vocab_size=30000):
    with open(f"{model_prefix}_training.txt", "w", encoding="utf-8") as f:
        for line in input_texts:
            f.write(line + "\\n")

    spm.SentencePieceTrainer.train(
        input=f"{model_prefix}_training.txt",
        model_prefix=model_prefix,
        vocab_size=vocab_size,
        character_coverage=1.0,
        model_type="unigram"
    )

    print(f"✅ SentencePiece model trained: {model_prefix}.model")

def load_tokenizer(model_path):
    sp = spm.SentencePieceProcessor()
    sp.load(model_path)
    return sp

def encode_and_pad(texts, sp, max_len):
    encoded = [sp.encode(text, out_type=int)[:max_len] for text in texts]
    padded = [seq + [0] * (max_len - len(seq)) if len(seq) < max_len else seq for seq in encoded]
    return torch.tensor(padded, dtype=torch.long)
''')


In [8]:
import sys
import os
import pandas as pd
import torch

# Add preprocessing module path
sys.path.append("/kaggle/working")

from TextSummarization.preprocessing.tokenizer import train_sentencepiece, load_tokenizer, encode_and_pad

# Create tokenizer dir
os.makedirs("TextSummarization/tokenizer", exist_ok=True)

# Load preprocessed data
df = pd.read_parquet("TextSummarization/data/processed/train_sample_50k.parquet")
df1 = pd.read_parquet("TextSummarization/data/processed/validation_sample.parquet")
df2 = pd.read_parquet("TextSummarization/data/processed/test_sample.parquet")

# Train SentencePiece tokenizer
train_sentencepiece(
    input_texts=df["clean_article"].tolist() + df["clean_summary"].tolist(),
    model_prefix="TextSummarization/tokenizer/spm",
    vocab_size=30000
)

# Load tokenizer
sp = load_tokenizer("TextSummarization/tokenizer/spm.model")

# Encode + pad
article_input = encode_and_pad(df["clean_article"], sp, max_len=512)
summary_input = encode_and_pad(df["clean_summary"], sp, max_len=128)


# Save tokenized tensors
torch.save(article_input, "TextSummarization/data/processed/train_article.pt")
torch.save(summary_input, "TextSummarization/data/processed/train_summary.pt")

article_input = encode_and_pad(df1["clean_article"], sp, max_len=512)
summary_input = encode_and_pad(df1["clean_summary"], sp, max_len=128)
torch.save(article_input, "TextSummarization/data/processed/val_article.pt")
torch.save(summary_input, "TextSummarization/data/processed/val_summary.pt")

article_input = encode_and_pad(df2["clean_article"], sp, max_len=512)
summary_input = encode_and_pad(df2["clean_summary"], sp, max_len=128)
torch.save(article_input, "TextSummarization/data/processed/test_article.pt")
torch.save(summary_input, "TextSummarization/data/processed/test_summary.pt")

print("✅ Tokenization and padding complete.")


sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: TextSummarization/tokenizer/spm_training.txt
  input_format: 
  model_prefix: TextSummarization/tokenizer/spm
  model_type: UNIGRAM
  vocab_size: 30000
  self_test_sample_size: 0
  character_coverage: 1
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 0
  bos_id: 1
  eos_id: 2
  pad_id: -1
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_su

✅ SentencePiece model trained: TextSummarization/tokenizer/spm.model
✅ Tokenization and padding complete.


In [8]:
import os
os.makedirs("TextSummarization/model", exist_ok=True)
with open("TextSummarization/model/cnn_summarizer.py", "w") as f:
    f.write('''import torch
import torch.nn as nn
import torch.nn.functional as F

class CNNEncoder(nn.Module):
    def __init__(self, vocab_size, embed_size, num_filters, kernel_sizes, padding_idx=0):
        super(CNNEncoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=padding_idx)
        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=embed_size, out_channels=num_filters, kernel_size=k, padding=k // 2)
            for k in kernel_sizes
        ])
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        # x: [batch_size, seq_len]
        x = self.embedding(x)              # [batch_size, seq_len, embed_size]
        x = x.transpose(1, 2)              # [batch_size, embed_size, seq_len]
        conv_outs = [F.relu(conv(x)) for conv in self.convs]  # list of [batch_size, num_filters, seq_len]
        out = torch.cat(conv_outs, dim=1)  # [batch_size, num_filters * len(kernel_sizes), seq_len]
        out = self.dropout(out)
        return out.transpose(1, 2)         # [batch_size, seq_len, hidden_size]


class CNNDecoder(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, kernel_size=3, padding_idx=0):
        super(CNNDecoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=padding_idx)
        self.conv1 = nn.Conv1d(embed_size, hidden_size, kernel_size, padding=kernel_size // 2)
        self.attn = nn.Linear(hidden_size, hidden_size)
        self.fc_out = nn.Linear(hidden_size * 2, vocab_size)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x, encoder_outputs):
        # x: [batch_size, tgt_seq_len]
        # encoder_outputs: [batch_size, src_seq_len, hidden_size]
        x = self.embedding(x)              # [batch_size, tgt_seq_len, embed_size]
        x = x.transpose(1, 2)              # [batch_size, embed_size, tgt_seq_len]
        x = F.relu(self.conv1(x))          # [batch_size, hidden_size, tgt_seq_len]
        x = x.transpose(1, 2)              # [batch_size, tgt_seq_len, hidden_size]

        # attention
        attn_scores = torch.bmm(x, encoder_outputs.transpose(1, 2))  # [batch_size, tgt_len, src_len]
        attn_weights = F.softmax(attn_scores, dim=-1)                # [batch_size, tgt_len, src_len]
        context = torch.bmm(attn_weights, encoder_outputs)          # [batch_size, tgt_len, hidden_size]

        combined = torch.cat((x, context), dim=-1)                   # [batch_size, tgt_len, hidden*2]
        output = self.fc_out(self.dropout(combined))                # [batch_size, tgt_len, vocab_size]
        return output


class CNNSummarizer(nn.Module):
    def __init__(self, vocab_size, embed_size=256, num_filters=128, kernel_sizes=[3,5,7], padding_idx=0):
        super(CNNSummarizer, self).__init__()
        self.encoder = CNNEncoder(vocab_size, embed_size, num_filters, kernel_sizes, padding_idx)
        self.decoder = CNNDecoder(vocab_size, embed_size, num_filters * len(kernel_sizes), kernel_size=3, padding_idx=padding_idx)

    def forward(self, src, tgt):
        encoder_outputs = self.encoder(src)
        output = self.decoder(tgt, encoder_outputs)
        return output
    def generate(self, input_tensor, max_length=128):
        with torch.no_grad():
            logits = self.forward(input_tensor)  # shape: (batch_size, seq_len, vocab_size)
            probs = torch.softmax(logits, dim=-1)
            predictions = torch.argmax(probs, dim=-1)  # shape: (batch_size, seq_len)
            return predictions[0].tolist()[:max_length]
    
''')

In [10]:
import os
os.makedirs("TextSummarization/training", exist_ok=True)
with open("TextSummarization/training/train_cnn.py", "w") as f:
    f.write('''import os
import sys
sys.path.append('/kaggle/working/TextSummarization')
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from model.cnn_summarizer import CNNSummarizer
from tqdm import tqdm

# Configs
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 32
epochs = 5
lr = 3e-4
vocab_size = 30000
pad_token_id = 0
checkpoint_path = "TextSummarization/checkpoints/cnn_checkpoint.pt"
best_model_path = "TextSummarization/checkpoints/cnn_best_model.pt"

# Make checkpoint directory
os.makedirs(os.path.dirname(checkpoint_path), exist_ok=True)

# Load data
train_src = torch.load("TextSummarization/data/processed/train_article.pt")
train_tgt = torch.load("TextSummarization/data/processed/train_summary.pt")
val_src = torch.load("TextSummarization/data/processed/val_article.pt")
val_tgt = torch.load("TextSummarization/data/processed/val_summary.pt")

train_dataset = TensorDataset(train_src, train_tgt)
val_dataset = TensorDataset(val_src, val_tgt)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

# Model, loss, optimizer
model = CNNSummarizer(vocab_size).to(device)
criterion = nn.CrossEntropyLoss(ignore_index=pad_token_id)
optimizer = optim.Adam(model.parameters(), lr=lr)

start_epoch = 1
best_val_loss = float("inf")

# 🔁 Load from checkpoint if exists
if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    start_epoch = checkpoint["epoch"] + 1
    best_val_loss = checkpoint["best_val_loss"]
    print(f"📦 Loaded checkpoint from epoch {checkpoint['epoch']}")

def train_one_epoch(model, dataloader):
    model.train()
    total_loss = 0
    for src, tgt in tqdm(dataloader, desc="🔁 Training"):
        src, tgt = src.to(device), tgt.to(device)
        decoder_input = tgt[:, :-1]
        target = tgt[:, 1:]

        output = model(src, decoder_input)
        output = output.reshape(-1, output.size(-1))
        target = target.reshape(-1)

        loss = criterion(output, target)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    return total_loss / len(dataloader)

def evaluate(model, dataloader):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for src, tgt in tqdm(dataloader, desc="🔍 Evaluating"):
            src, tgt = src.to(device), tgt.to(device)
            decoder_input = tgt[:, :-1]
            target = tgt[:, 1:]

            output = model(src, decoder_input)
            output = output.reshape(-1, output.size(-1))
            target = target.reshape(-1)

            loss = criterion(output, target)
            total_loss += loss.item()
    return total_loss / len(dataloader)

# 🏋️ Training loop
for epoch in range(start_epoch, epochs + 1):
    print(f"🌟 Epoch {epoch}/{epochs}")
    train_loss = train_one_epoch(model, train_loader)
    val_loss = evaluate(model, val_loader)

    print(f"✅ Epoch {epoch} — Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    # 💾 Save checkpoint
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "best_val_loss": best_val_loss
    }, checkpoint_path)
    print(f"📌 Checkpoint saved: {checkpoint_path}")

    # 🔐 Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), best_model_path)
        print(f"🌟 Best model updated: {best_model_path}")
''')

In [11]:
!python TextSummarization/training/train_cnn.py

🌟 Epoch 1/5
🔍 Evaluating: 100%|██████████████████████████| 418/418 [00:28<00:00, 14.57it/s]
✅ Epoch 1 — Train Loss: 3.1042 | Val Loss: 0.4951
📌 Checkpoint saved: TextSummarization/checkpoints/cnn_checkpoint.pt
🌟 Best model updated: TextSummarization/checkpoints/cnn_best_model.pt
🌟 Epoch 2/5
🔍 Evaluating: 100%|██████████████████████████| 418/418 [00:28<00:00, 14.48it/s]
✅ Epoch 2 — Train Loss: 0.2446 | Val Loss: 0.1170
📌 Checkpoint saved: TextSummarization/checkpoints/cnn_checkpoint.pt
🌟 Best model updated: TextSummarization/checkpoints/cnn_best_model.pt
🌟 Epoch 3/5
🔍 Evaluating: 100%|██████████████████████████| 418/418 [00:30<00:00, 13.67it/s]
✅ Epoch 3 — Train Loss: 0.0608 | Val Loss: 0.0610
📌 Checkpoint saved: TextSummarization/checkpoints/cnn_checkpoint.pt
🌟 Best model updated: TextSummarization/checkpoints/cnn_best_model.pt
🌟 Epoch 4/5
🔍 Evaluating: 100%|██████████████████████████| 418/418 [00:31<00:00, 13.29it/s]
✅ Epoch 4 — Train Loss: 0.0237 | Val Loss: 0.0427
📌 Checkpoint saved

In [ ]:
!pip install rouge-score evaluate bert-score textstat --quiet

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:000:00:0100:01
   ━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.1/664.8 MB 178.2 MB/s eta 0:00:04

In [ ]:
import os, sys, torch, pandas as pd, sentencepiece as spm
from tqdm import tqdm
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
from bert_score import score as bert_score
import textstat, spacy
from collections import Counter

sys.path.append('/kaggle/working/TextSummarization')
from TextSummarization.model.cnn_summarizer import CNNSummarizer

# Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_path = "TextSummarization/checkpoints/cnn_best_model.pt"
tokenizer_path = "TextSummarization/tokenizer/spm.model"
max_input_len = 512
max_output_len = 128
results_path = "TextSummarization/results"
os.makedirs(results_path, exist_ok=True)

# Load SentencePiece tokenizer
sp = spm.SentencePieceProcessor()
sp.load(tokenizer_path)

# Load model
model = CNNSummarizer(
    vocab_size=30000, embed_size=256, num_filters=128,
    kernel_sizes=[3, 5, 7], padding_idx=0
)
model.load_state_dict(torch.load(model_path, map_location=device))
model.to(device)
model.eval()

# Load test data
test_df = pd.read_parquet("TextSummarization/data/processed/test_sample.parquet")[:128]
references = test_df["clean_summary"].tolist()
articles = test_df["clean_article"].tolist()

# Generate summaries
generated_summaries = []
for article in tqdm(articles, desc="🚀 Generating summaries"):
    input_ids = sp.encode(article, out_type=int)[:max_input_len]
    if not input_ids:
        generated_summaries.append("")
        continue
    input_tensor = torch.tensor(input_ids, dtype=torch.long).unsqueeze(0).to(device)
    with torch.no_grad():
        output_ids = model.generate(input_tensor, max_length=max_output_len)
    generated_summaries.append(sp.decode(output_ids))

# Setup Scorers
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
nlp = spacy.load("en_core_web_sm")

# Helper: cohesion
def cohesion_score(text):
    doc = nlp(text)
    lemmas = [t.lemma_.lower() for t in doc if not t.is_stop and not t.is_punct]
    pronouns = [t.text.lower() for t in doc if t.pos_ == "PRON"]
    if not lemmas: return 0.0
    lemma_freq = Counter(lemmas)
    repeated = sum(1 for count in lemma_freq.values() if count > 1)
    return round((repeated + len(pronouns)) / len(lemmas), 4)

# Evaluation metrics
rouge1, rouge2, rougeL = [], [], []
meteor_scores = []
precision_scores, recall_scores, f1_scores = [], [], []
cohesion_scores = []
readability, fog, ari, dale_chall = [], [], [], []

# Compute metrics
for ref, gen in zip(references, generated_summaries):
    scores = scorer.score(ref, gen)
    rouge1.append(scores["rouge1"].fmeasure)
    rouge2.append(scores["rouge2"].fmeasure)
    rougeL.append(scores["rougeL"].fmeasure)

    meteor_scores.append(meteor_score([ref.split()], gen.split()))

    ref_tokens, gen_tokens = ref.split(), gen.split()
    common = set(ref_tokens) & set(gen_tokens)
    p = len(common) / len(gen_tokens) if gen_tokens else 0
    r = len(common) / len(ref_tokens) if ref_tokens else 0
    f1 = 2 * p * r / (p + r) if (p + r) else 0
    precision_scores.append(p)
    recall_scores.append(r)
    f1_scores.append(f1)

    cohesion_scores.append(cohesion_score(gen))
    readability.append(textstat.flesch_reading_ease(gen))
    fog.append(textstat.gunning_fog(gen))
    ari.append(textstat.automated_readability_index(gen))
    dale_chall.append(textstat.dale_chall_readability_score(gen))

# BERTScore
P, R, F1 = bert_score(generated_summaries, references, lang="en", verbose=True)

# Average helper
def avg(x): return sum(x) / len(x)

# Print all results
print("\n📊 Evaluation Summary:")
print(f"ROUGE-1 F1:           {avg(rouge1):.4f}")
print(f"ROUGE-2 F1:           {avg(rouge2):.4f}")
print(f"ROUGE-L F1:           {avg(rougeL):.4f}")
print(f"METEOR:               {avg(meteor_scores):.4f}")
print(f"BERTScore Precision:  {P.mean().item():.4f}")
print(f"BERTScore Recall:     {R.mean().item():.4f}")
print(f"BERTScore F1:         {F1.mean().item():.4f}")
print(f"Token Precision:      {avg(precision_scores):.4f}")
print(f"Token Recall:         {avg(recall_scores):.4f}")
print(f"Token F1:             {avg(f1_scores):.4f}")
print(f"Cohesion Score:       {avg(cohesion_scores):.4f}")
print(f"Readability (Flesch): {avg(readability):.2f}")
print(f"Gunning Fog:          {avg(fog):.2f}")
print(f"ARI:                  {avg(ari):.2f}")
print(f"Dale-Chall:           {avg(dale_chall):.2f}")

# Save output summaries
pd.DataFrame({
    "article": articles,
    "reference": references,
    "generated": generated_summaries
}).to_csv(f"{results_path}/generated_summaries.csv", index=False)
print(f"\n✅ Results saved to {results_path}/generated_summaries.csv")

# Optional: QuestEval (install + use)
# from questeval.questeval_metric import QuestEval
# questeval = QuestEval()
# quest_scores = questeval.corpus_questeval(hypotheses=generated_summaries, sources=articles, list_references=references)
# print(f"QuestEval F1: {quest_scores['corpus_score']:.4f}")
